# CUDA × LLM 推理教程 — Colab 一键启动

运行步骤：
1. Runtime → Change runtime type → **T4 GPU**（或更高）
2. 从上到下依次运行所有 cell
3. 训练/推理示例会自动落在 `/content/ops` 目录

**预计时间**：环境准备 ~2 分钟；编译全部 14 章 ~5 分钟。

In [ ]:
# 1) 检查 GPU
!nvidia-smi | head -20
!nvcc --version

Wed Jul  8 07:29:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2) 克隆仓库
import os
if not os.path.exists('/content/ops'):
    # TODO: replace with your fork URL after pushing
    !git clone https://github.com/hswsp/learn-cuda-from-scratch.git /content/ops
%cd /content/ops

Cloning into '/content/ops'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 193 (delta 28), reused 185 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 258.02 KiB | 1.42 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/ops


In [ ]:
# 3) 选择 GPU 架构
# Colab Free T4 → sm_75, A100 → sm_80, L4 → sm_89
import subprocess
out = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True).decode().strip()
print('detected GPU:', out)
ARCH = 'sm_75' if 'T4' in out else ('sm_89' if 'L4' in out else 'sm_80')
print('using ARCH =', ARCH)

detected GPU: Tesla T4
using ARCH = sm_75


In [ ]:
# 4) 编译并跑通第 2 章 Hello CUDA
!make -C code/ch02_hello ARCH={ARCH} clean all run

make: Entering directory '/content/ops/code/ch02_hello'
rm -f hello host_device_memcpy error_handling *.o
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr hello.cu -o hello
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr host_device_memcpy.cu -o host_device_memcpy
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr error_handling.cu -o error_handling
--- hello ---
--- launching 2 blocks x 4 threads ---
hello from thread (0, 0) of block (1, 0)
hello from thread (1, 0) of block (1, 0)
hello from thread (2, 0) of block (1, 0)
hello from thread (3, 0) of block (1, 0)
hello from thread (0, 0) of block (0, 0)
hello from thread (1, 0) of block (0, 0)
hello from thread (2, 0) of block (0, 0)
hello from thread (3, 0) of block (0, 0)
--- launching 1 block x dim3(2,2) threads ---
hello from thread (0, 0) of block (0, 0)
hello from thread (1, 0) of block (0, 0)
hello from thread (0, 1) of block (0, 0)
hello from thread (1, 1) of block (0, 0)
done

In [ ]:
# 5) 跑核心章节示例（Ch6 Tile / Ch9 GEMM / Ch12 FlashAttention）
for ch in ['ch06_tile', 'ch09_gemm', 'ch12_flashattn']:
    print(f'==> {ch}')
    !make -C code/{ch} ARCH={ARCH} run | tail -30

==> ch06_tile
make: Entering directory '/content/ops/code/ch06_tile'
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr matmul_naive.cu -o matmul_naive
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr matmul_tiled.cu -o matmul_tiled
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr transpose.cu -o transpose
[matmul_naive] PASS  max_abs=1.048e-09  max_rel=1.142e-02  l2=1.476e-10
MNK=512x512x512  1.212 ms  221.5 GFLOPS
[matmul_tiled] PASS  max_abs=1.048e-09  max_rel=1.142e-02  l2=1.476e-10
tiled  MNK=512x512x512  0.394 ms  681.2 GFLOPS
  naive                PASS  1.514 ms  88.7 GB/s
  shared               PASS  1.125 ms  119.3 GB/s
  padded               PASS  0.667 ms  201.3 GB/s
make: Leaving directory '/content/ops/code/ch06_tile'
==> ch09_gemm
gemm_cublas.cu(44): error: identifier "matmul_ops" is undefined
                  M, N, K, t.ms(), gflops(matmul_ops(M, N, K), t.ms()));
                                          ^

gemm_cubla

In [ ]:
# 6) 下载 GPT-2 small 权重（Capstone 用，~250MB）
!pip install -q transformers torch
!python data/download_gpt2.py --out data/gpt2-small.bin

In [ ]:
# 7) 跑 Capstone：用自己写的 kernel 做 GPT-2 推理
!make -C code/ch14_mini_llm ARCH={ARCH} clean all
!./code/ch14_mini_llm/mini_llm --weights data/gpt2-small.bin --prompt 'Once upon a time'

# 导论与环境

In [ ]:
!make -C code/ch01_intro ARCH=sm_75 run

make: Entering directory '/content/ops/code/ch01_intro'
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr device_query.cu -o device_query
nvcc -O2 -std=c++17 -arch=sm_75 -lineinfo --expt-relaxed-constexpr bandwidth_estimate.cu -o bandwidth_estimate
--- device_query ---
CUDA driver:  13.0
CUDA runtime: 12.8
Devices:      1

GPU 0: Tesla T4
  Compute capability : 7.5  (sm_75, Turing)
  SM count           : 40
  Max threads / block: 1024
  Max threads / SM   : 1024  (= 32 warps)
  Warp size          : 32
  Registers / SM     : 65536 (32-bit)
  Shared mem / block : 49152 B
  Shared mem / SM    : 65536 B
  Global memory      : 14.56 GiB
  L2 cache           : 4096 KiB
  Memory clock       : 5.0 GHz
  Memory bus width   : 256 bit
  Peak bandwidth     : 320.1 GB/s
  Clock rate         : 1.59 GHz
  ECC enabled        : yes
  Unified addressing : yes
  Concurrent kernels : yes


--- bandwidth_estimate ---
payload = 64 MiB, iters = 5

transfer                               ms   

In [ ]:
%cd code/ch01_intro
!make ARCH=sm_75    # T4
!./device_query

/content/ops/code/ch01_intro
make: Nothing to be done for 'all'.
CUDA driver:  13.0
CUDA runtime: 12.8
Devices:      1

GPU 0: Tesla T4
  Compute capability : 7.5  (sm_75, Turing)
  SM count           : 40
  Max threads / block: 1024
  Max threads / SM   : 1024  (= 32 warps)
  Warp size          : 32
  Registers / SM     : 65536 (32-bit)
  Shared mem / block : 49152 B
  Shared mem / SM    : 65536 B
  Global memory      : 14.56 GiB
  L2 cache           : 4096 KiB
  Memory clock       : 5.0 GHz
  Memory bus width   : 256 bit
  Peak bandwidth     : 320.1 GB/s
  Clock rate         : 1.59 GHz
  ECC enabled        : yes
  Unified addressing : yes
  Concurrent kernels : yes



In [ ]:
!./bandwidth_estimate

payload = 64 MiB, iters = 5

transfer                               ms         GB/s
------------------------------------------------------
H2D pageable                       14.588         4.60
D2H pageable                       13.466         4.98
H2D pinned                          5.446        12.32
D2H pinned                          5.122        13.10
D2D                                 0.566       118.67


In [12]:
!for mb in 1 4 16 64 256; do echo "=== --MB=$mb ===" && ./bandwidth_estimate --MB=$mb; done

=== --MB=1 ===
payload = 1 MiB, iters = 5

transfer                               ms         GB/s
------------------------------------------------------
H2D pageable                        0.333         3.15
D2H pageable                        0.252         4.16
H2D pinned                          0.098        10.67
D2H pinned                          0.092        11.36
D2D                                 0.011        93.14
=== --MB=4 ===
payload = 4 MiB, iters = 5

transfer                               ms         GB/s
------------------------------------------------------
H2D pageable                        0.997         4.21
D2H pageable                        0.836         5.02
H2D pinned                          0.351        11.93
D2H pinned                          0.329        12.76
D2D                                 0.040       104.34
=== --MB=16 ===
payload = 16 MiB, iters = 5

transfer                               ms         GB/s
--------------------------------------------

In [13]:
!nvidia-smi topo -m       # 看 GPU 之间的连接拓扑
# 输出 NV4 / NV8 = NVLink (好); SYS / PHB = 走 PCIe (慢)

	GPU0	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	0-1	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


In [14]:
# 实时刷新基本状态 (温度、功耗、显存、利用率)
!nvidia-smi -l 1

# 进程级显存占用 + GPU 利用率
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

# 持续监控的工程化方案 (NVIDIA DCGM)
!dcgmi dmon -e 1001,1002,1003,203,150
#  1001 = GR engine active   1002 = SM active   1003 = SM occupancy
#   203 = mem util           150  = mem copy util

Wed Jul  8 08:55:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----